# 01 – Join OSM vet clinics with LOR districts & neighborhoods

This notebook:

1. Loads the latest OSM vet clinics snapshot produced in `00_vet_clinics_osmnx_download.ipynb`.
2. Loads the official LOR / Ortsteile polygons for Berlin.
3. Performs a spatial join to assign each clinic to:
   - LOR polygon (`lor_id`)
   - district and neighborhood names
   - district and neighborhood IDs using the shared mapping pattern.
4. Exports a v0 dataset used as input to the cleaning step (v1).

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
if (ROOT / "veterinary_clinics" / "sources").exists():
    os.chdir(ROOT / "veterinary_clinics")
    print("Changed CWD to:", Path.cwd())
else:
    print("Current CWD:", ROOT)
    print("Assuming this notebook already runs inside `veterinary_clinics`.")

import geopandas as gpd
import pandas as pd

Current CWD: /Users/jorge/Projects/layered-populate-data-pool-da/veterinary_clinics
Assuming this notebook already runs inside `veterinary_clinics`.


## 1. Load OSM vet clinics and LOR polygons

In [2]:
OSM_GEOJSON_PATH = Path("sources/osmnx_berlin_vet_clinics_latest.geojson")
LOR_GEOJSON_PATH = Path("sources/raw_berlin_lor_ortsteile.geojson")

gdf_vets = gpd.read_file(OSM_GEOJSON_PATH)
gdf_lor = gpd.read_file(LOR_GEOJSON_PATH)

print("OSM vets:", gdf_vets.shape)
print("LOR polygons:", gdf_lor.shape)

gdf_vets.head()

OSM vets: (175, 22)
LOR polygons: (96, 8)


,element,id,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,...,website,contact:website,opening_hours,operator,wheelchair,wheelchair:description,emergency,lat,lon,geometry
0,node,268917040,node/268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961,Berlin,None,None,...,None,None,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",None,no,None,None,52.495684,13.405233,POINT (13.40523 52.49568)
1,node,299795048,node/299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125,Berlin,+49 30 9437820,None,...,http://www.tierarztpraxis-hartwig.de/,None,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",None,limited,None,None,52.606286,13.479555,POINT (13.47955 52.60629)
2,node,347294456,node/347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207,Berlin,+49 30 7738321,None,...,https://www.tierarztpraxis-soerensen.de/,None,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",None,yes,None,None,52.429722,13.320133,POINT (13.32013 52.42972)
3,node,394867279,node/394867279,Tierarztpraxis Jeanette Koepsel,None,None,None,None,None,None,...,None,None,None,None,None,None,None,52.535199,13.270573,POINT (13.27057 52.5352)
4,node,411550894,node/411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621,Berlin,+49 30 53018585,None,...,https://www.tierarzt-kaulsdorf.de/,None,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Dr. Berit Miels;Dr. Mathias Kochert,None,None,None,52.509511,13.589635,POINT (13.58964 52.50951)


## 2. Check and align CRS

In [3]:
print("Vet CRS:", gdf_vets.crs)
print("LOR CRS:", gdf_lor.crs)

if gdf_lor.crs != gdf_vets.crs:
    gdf_lor = gdf_lor.to_crs(gdf_vets.crs)
    print("Reprojected LOR to vets CRS:", gdf_lor.crs)

Vet CRS: EPSG:4326
LOR CRS: EPSG:4326


## 3. Prepare LOR subset and rename columns

We keep relevant fields and align naming with the shared pattern:

- `gml_id` → `lor_id`
- `BEZIRK` → `district`
- `OTEIL` → `neighborhood`
- `spatial_name` → `neighborhood_id`

In [4]:
lor_cols = ["gml_id", "BEZIRK", "spatial_name", "OTEIL", "geometry"]
gdf_lor_subset = gdf_lor[lor_cols].copy()

gdf_lor_subset = gdf_lor_subset.rename(
    columns={
        "gml_id": "lor_id",
        "BEZIRK": "district",
        "OTEIL": "neighborhood",
        "spatial_name": "neighborhood_id",
    }
)

gdf_lor_subset.head()

,lor_id,district,neighborhood_id,neighborhood,geometry
0,re_ortsteil.0101,Mitte,0101,Mitte,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,Mitte,0102,Moabit,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,Mitte,0103,Hansaviertel,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,Mitte,0104,Tiergarten,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,Mitte,0105,Wedding,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


In [5]:
# Ensure neighborhood_id is a 4-digit string (leading zeros preserved)
gdf_lor_subset["neighborhood_id"] = (
    gdf_lor_subset["neighborhood_id"]
    .astype("string")      # convert from numeric to string
    .str.strip()
    .str.zfill(4)          # pad to 4 characters (e.g. "303" -> "0303")
)

## 4. Spatial join: assign LOR district & neighborhood to each clinic

In [6]:
gdf_vets_lor = gpd.sjoin(
    gdf_vets,
    gdf_lor_subset,
    how="left",
    predicate="within",
)

if "index_right" in gdf_vets_lor.columns:
    gdf_vets_lor = gdf_vets_lor.drop(columns=["index_right"])

gdf_vets_lor[["name", "district", "neighborhood", "neighborhood_id"]].head()

,name,district,neighborhood,neighborhood_id
0,Tierarztpraxis am Urban,Friedrichshain-Kreuzberg,Kreuzberg,0202
1,Dr. med. vet. Elke Hartwig,Pankow,Karow,0305
2,Tierarztpraxis Dr. Bernhard Sörensen,Steglitz-Zehlendorf,Lichterfelde,0602
3,Tierarztpraxis Jeanette Koepsel,Spandau,Siemensstadt,0503
4,Kleintierarztpraxis Berlin Kaulsdorf,Marzahn-Hellersdorf,Kaulsdorf,1003


In [7]:
gdf_vets_lor["neighborhood_id"] = (
    gdf_vets_lor["neighborhood_id"].astype("string")
)

gdf_vets_lor["neighborhood_id"].str.len().value_counts()

neighborhood_id
4    175
Name: count, dtype: Int64

## 5. District mapping – add `district_id`

We map Berlin districts to the official district IDs, following
the same convention as the other POI layers.

In [8]:
district_mapping = {
    "Mitte": "11001001",
    "Friedrichshain-Kreuzberg": "11002002",
    "Pankow": "11003003",
    "Charlottenburg-Wilmersdorf": "11004004",
    "Spandau": "11005005",
    "Steglitz-Zehlendorf": "11006006",
    "Tempelhof-Schöneberg": "11007007",
    "Neukölln": "11008008",
    "Treptow-Köpenick": "11009009",
    "Marzahn-Hellersdorf": "11010010",
    "Lichtenberg": "11011011",
    "Reinickendorf": "11012012",
}

gdf_vets_lor["district_id"] = (
    gdf_vets_lor["district"]
    .astype("string")
    .map(district_mapping)
)

gdf_vets_lor[["district", "district_id"]].drop_duplicates().sort_values("district").head(20)

,district,district_id
19,Charlottenburg-Wilmersdorf,11004004
0,Friedrichshain-Kreuzberg,11002002
16,Lichtenberg,11011011
4,Marzahn-Hellersdorf,11010010
15,Mitte,11001001
7,Neukölln,11008008
1,Pankow,11003003
11,Reinickendorf,11012012
3,Spandau,11005005
2,Steglitz-Zehlendorf,11006006


## 6. Build v0 dataset (OSM + LOR context)

We keep OSM identifiers, attributes, geometry, and LOR context,
and export `vets_osm_berlin_with_lor_latest_v0.csv`, which will be
the input for the cleaning notebook.

In [9]:
cols_v0 = [
    "id",               
    "element",
    "source_osm_id",
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "phone",
    "contact:phone",
    "email",
    "contact:email",
    "website",
    "contact:website",
    "opening_hours",
    "operator",
    "wheelchair",
    "wheelchair:description",
    "emergency",
    "lat",
    "lon",
    "geometry",
    "lor_id",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
]

cols_v0 = [c for c in cols_v0 if c in gdf_vets_lor.columns]

df_v0 = gdf_vets_lor[cols_v0].copy()

from pathlib import Path

output_v0_csv = Path("cache/vets_osm_berlin_with_lor_latest_v0.csv")
output_v0_csv.parent.mkdir(parents=True, exist_ok=True)

df_v0.to_csv(output_v0_csv, index=False)

print("Exported v0 dataset to:")
print(" -", output_v0_csv)
print("Shape:", df_v0.shape)

df_v0.head()

Exported v0 dataset to:
 - cache/vets_osm_berlin_with_lor_latest_v0.csv
Shape: (175, 27)


,id,element,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,...,wheelchair:description,emergency,lat,lon,geometry,lor_id,district,district_id,neighborhood,neighborhood_id
0,268917040,node,node/268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961,Berlin,None,None,...,None,None,52.495684,13.405233,POINT (13.40523 52.49568),re_ortsteil.0202,Friedrichshain-Kreuzberg,11002002,Kreuzberg,0202
1,299795048,node,node/299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125,Berlin,+49 30 9437820,None,...,None,None,52.606286,13.479555,POINT (13.47955 52.60629),re_ortsteil.0305,Pankow,11003003,Karow,0305
2,347294456,node,node/347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207,Berlin,+49 30 7738321,None,...,None,None,52.429722,13.320133,POINT (13.32013 52.42972),re_ortsteil.0602,Steglitz-Zehlendorf,11006006,Lichterfelde,0602
3,394867279,node,node/394867279,Tierarztpraxis Jeanette Koepsel,None,None,None,None,None,None,...,None,None,52.535199,13.270573,POINT (13.27057 52.5352),re_ortsteil.0503,Spandau,11005005,Siemensstadt,0503
4,411550894,node,node/411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621,Berlin,+49 30 53018585,None,...,None,None,52.509511,13.589635,POINT (13.58964 52.50951),re_ortsteil.1003,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003
